In [ ]:
!uv add "langgraph==1.2.9" "langchain-groq==1.1.3" "python-dotenv>=1.0.1" "ipykernel>=7.0.0"

In [ ]:
from typing import List, Optional, TypedDict

In [ ]:
class AgentState(TypedDict, total=False):
    user_input: str
    task_list: List[str]
    final_output: Optional[str]

In [ ]:
import os

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise RuntimeError("GROQ_API_KEY is not available. Rebuild the Codespace after adding the secret.")

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(groq_api_key=groq_api_key, model_name="llama-3.3-70b-versatile")
llm

In [ ]:
def planner_node(state: dict) -> dict:
    prompt = f"Break this task into 2-3 steps: {state['user_input']}"
    response  = llm.invoke(prompt)  # Assuming you have a function defined to call the LLM
    steps = response.content
    step_lines = [line.strip("- ").strip() for line in steps.split("\n") if line.strip()]
    print("📌 Planner temp Output:", step_lines[0] if step_lines else "No steps generated")
    print("📌 Planner Output:", step_lines)
    return {"task_list": step_lines}

In [ ]:
def executor_node(state: dict) -> dict:
    steps = state.get("task_list", [])
    final_output = " -> ".join(steps) + " -> Done!"
    return {"final_output": final_output}

In [ ]:
import langgraph.graph

In [ ]:
from langgraph.graph import StateGraph

In [ ]:
graph = StateGraph(state_schema=AgentState)

In [ ]:
graph.add_node("planner", planner_node)
graph.add_node("executor", executor_node)

graph.set_entry_point("planner")
graph.add_edge("planner", "executor")
graph.set_finish_point("executor")

In [ ]:
graph_compiled = graph.compile()

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph_compiled.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
state = {"user_input": "How do solar panels work?"}
final_state = graph_compiled.invoke(state)

In [ ]:
print(final_state["final_output"])